# 24회차 실습: 다중 선형회귀, 데이터 나누기, 표준화, KNN 준비

위에서부터 한 칸씩 `Shift + Enter`로 실행하며 따라와요. 칸 위 **나와야 하는 것**과 내 결과가 같으면 맞게 따라온 거예요.

- 프로젝트: 22회차의 `~/dev/my-pandas` (pandas, scikit-learn, seaborn, matplotlib이 들어 있어요)
- 오늘은 새로 설치할 것도, CSV 파일도 없어요. 7절의 얼굴 사진은 첫 실행 때 인터넷에서 약 1.4MB를 받아 `datasets` 폴더에 저장해요.

## 1. 준비: 도구 불러오기

**나와야 하는 것:** `pandas 3.1.1 sklearn 1.9.1 seaborn 0.13.2`처럼 버전 세 개 (숫자는 조금 달라도 괜찮아요). `ModuleNotFoundError`가 나오면 터미널에서 `uv add scikit-learn seaborn`을 하고, VS Code 위쪽 **Restart**로 커널을 다시 켜요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 한글 폰트 설정
# plt.rcParams["font.family"] = "Malgun Gothic"   # 윈도우 사용자
plt.rcParams["font.family"] = "AppleGothic"     # 맥 사용자

# 마이너스 기호 깨짐 막기
plt.rcParams["axes.unicode_minus"] = False

print("pandas", pd.__version__,
      "sklearn", sklearn.__version__,
      "seaborn", sns.__version__)

## 2. 다중 선형회귀: 특징이 셋이면 기울기도 셋

### 데이터와 입력, 정답 나누기

어제 7절의 학생 10명 데이터예요.

**나와야 하는 것:** `(10, 3)`과 `(10,)`. 입력은 10행 3열 표, 정답은 값 10개짜리 열 하나예요.

In [ ]:
df = pd.DataFrame({
    "study_hours":    [1, 2, 3, 4, 5, 2, 3, 5, 4, 1],
    "sleep_hours":    [6, 8, 5, 7, 8, 5, 8, 6, 6, 8],
    "practice_count": [10, 15, 20, 25, 30, 12, 28, 22, 32, 8],
    "score":          [47, 58, 60, 76, 87, 49, 73, 77, 79, 52]
})

# 입력으로 쓸 열 세 개
features = ["study_hours", "sleep_hours", "practice_count"]

# 입력(2차원)
X = df[features]
# 정답(1차원)
y = df["score"]

print(X.shape)
print(y.shape)

### 학습하고 가중치, 절편 보기

**나와야 하는 것:** `가중치 [5.80222725 2.68100001 0.63219852]`, `절편 17.66020808020...` (끝자리는 조금 달라도 괜찮아요)

In [ ]:
model = LinearRegression()
model.fit(X, y)

# 특징 셋이라 가중치도 셋 (features 순서 그대로)
print("가중치", model.coef_)
print("절편", model.intercept_)

### 가중치를 막대로

**나와야 하는 것:** 가로 막대 세 개. `study_hours`가 가장 길고(약 5.8), `practice_count`가 가장 짧아요(약 0.6).

In [ ]:
coef_df = pd.DataFrame({
    "feature": features,
    "weight": model.coef_
})

plt.figure(figsize=(6, 2))

# 가로 막대: 세로축에 특징 이름
plt.barh(
    coef_df["feature"],
    coef_df["weight"],
    color="blue"
)
plt.show()

### 새 학생 예측과 손 계산

공부 4시간, 수면 7시간, 문제 30개인 학생이에요.

**나와야 하는 것:** `예측 성적: 78.60점`, 그리고 `직접 계산: 78.6021`과 `predict 계산: 78.6021`이 같아요.

In [ ]:
new_data = pd.DataFrame({
    "study_hours": [4],
    "sleep_hours": [7],
    "practice_count": [30]
})

prediction = model.predict(new_data)
print(f"예측 성적: {prediction[0]:.2f}점")

In [ ]:
# 가중치 셋을 차례로 꺼낸다
w1, w2, w3 = model.coef_
b = model.intercept_

# 회귀식: w1 x1 + w2 x2 + w3 x3 + b
result = w1 * 4 + w2 * 7 + w3 * 30 + b

print(f"직접 계산: {result:.4f}")
print(f"predict 계산: {prediction[0]:.4f}")

## 3. 학습한 데이터로 채점하면 MAE 1.17, R² 0.99

### 실제 점수 옆에 예측값과 오차 붙이기

**나와야 하는 것:** 열이 6개인 10행 표. 0번 학생은 `47  45.87  1.13`, 1번 학생은 `58  60.20  -2.20`으로 끝나요.

In [ ]:
y_pred = model.predict(X)

# 원본을 건드리지 않게 복사본에 열을 더한다
result_df = df.copy()

# 없는 열 이름에 넣으면 새 열이 생긴다
result_df["예측값"] = y_pred
result_df["예측오차"] = y - y_pred

result_df.round(2)

### 평가 지표

**나와야 하는 것:** `MAE: 1.1739`, `MSE: 1.8252`, `R2 : 0.9900`

In [ ]:
mae = mean_absolute_error(y, y_pred)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R2 : {r2:.4f}")

## 4. 데이터 나누기: 처음 보는 3명으로 채점하면 MAE 3.09

### 7명은 학습용, 3명은 시험용

**나와야 하는 것:** `X_train (7, 3)`, `X_test (3, 3)`, `y_train (7,)`, `y_test (3,)`, 그리고 시험용 학생 번호 `[8, 1, 5]`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

# 시험용으로 뽑힌 학생의 행 번호
print(list(X_test.index))

### 학습용으로만 배우고 시험용으로 채점

**나와야 하는 것:** `MAE: 3.0870`, `MSE: 10.6380`, `R2: 0.9327`. 3절(학습한 데이터로 채점)보다 나빠지는 것이 정상이에요.

In [ ]:
test_model = LinearRegression()

# 70% 데이터로 학습
test_model.fit(X_train, y_train)

# 학습하지 않은 30%로 예측
test_pred = test_model.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, test_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, test_pred):.4f}")
print(f"R2: {r2_score(y_test, test_pred):.4f}")

## 5. 표준화: 단위가 다른 특징을 평균 0, 표준편차 1로 맞춘다

### 특징마다 범위가 다르다

**나와야 하는 것:** `min`, `max` 줄에서 공부 시간 1~5, 수면 시간 5~8, 문제 풀이 8~32

In [ ]:
# 기초 통계: 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값
X.describe().round(2)

### StandardScaler

**나와야 하는 것:** 7행 3열 표. 0번 행이 `-1.38  -0.76  -1.32`예요.

In [ ]:
scaler = StandardScaler()

# 학습 데이터로 평균, 표준편차를 재고(fit) 바로 바꾼다(transform)
X_train_scaled = scaler.fit_transform(X_train)

# 시험 데이터는 학습 데이터의 평균, 표준편차로 바꾸기만 한다
X_test_scaled = scaler.transform(X_test)

scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=features
)

scaled_df.round(2)

### 재 둔 평균과 표준편차, 바뀐 뒤의 평균과 표준편차

**나와야 하는 것:** 평균 `[ 3.14  6.86 20.43]`, 표준편차 `[1.55 1.12 7.89]`, 바뀐 뒤 평균은 모두 `0`, 표준편차는 모두 `1`

In [ ]:
print("평균", scaler.mean_.round(2))
print("표준편차", scaler.scale_.round(2))

# 바뀐 학습 데이터의 열마다 평균과 표준편차
print("바뀐 뒤 평균", X_train_scaled.mean(axis=0).round(2))
print("바뀐 뒤 표준편차", X_train_scaled.std(axis=0).round(2))

### 0번 학생의 공부 시간을 손으로

**나와야 하는 것:** `-1.38`. `(1 - 3.14) ÷ 1.55`예요.

In [ ]:
first = X_train.iloc[0]["study_hours"]
z = (first - scaler.mean_[0]) / scaler.scale_[0]
print(round(z, 2))

## 6. 경사하강법으로 다중 회귀: 표준화한 데이터로 걷는다

### fit으로 한 번에

**나와야 하는 것:** `MAE: 2.8169`, `MSE: 8.5966`, `R2: 0.9456`, `멈춘 epoch: 197`

In [ ]:
sgd = SGDRegressor(
    loss="squared_error",
    learning_rate="constant",
    eta0=0.01,
    max_iter=3000,
    random_state=42
)

# 표준화한 학습 데이터로 학습
sgd.fit(X_train_scaled, y_train)

# 표준화한 시험 데이터로 예측
sgd_pred = sgd.predict(X_test_scaled)

print(f"MAE: {mean_absolute_error(y_test, sgd_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, sgd_pred):.4f}")
print(f"R2: {r2_score(y_test, sgd_pred):.4f}")

# 더 나아지지 않아 스스로 멈춘 바퀴 수
print("멈춘 epoch:", sgd.n_iter_)

### 표준화하지 않으면

수업에 없던 칸이에요. 같은 설정으로 원래 숫자를 그대로 넣어 봐요.

**나와야 하는 것:** 가중치가 `3.37e+10`처럼 백억 단위로 튀고, 예측도 터무니없는 값이에요.

In [ ]:
raw = SGDRegressor(
    loss="squared_error",
    learning_rate="constant",
    eta0=0.01,
    max_iter=3000,
    random_state=42
)
raw.fit(X_train, y_train)

print("가중치", raw.coef_)
print("예측", raw.predict(X_test))

### 500 epoch 기록

**나와야 하는 것:** 마지막 줄 `500  9.93  3.62  3.29  67.42  0.14`

In [ ]:
sgd = SGDRegressor(
    loss="squared_error",
    learning_rate="constant",
    eta0=0.01,
    random_state=42
)

history = []

for epoch in range(500):
    # 1 epoch 학습
    sgd.partial_fit(X_train_scaled, y_train)
    pred = sgd.predict(X_train_scaled)

    history.append({
        "epoch": epoch + 1,
        "w1": sgd.coef_[0],
        "w2": sgd.coef_[1],
        "w3": sgd.coef_[2],
        "bias": sgd.intercept_[0],
        "mse": mean_squared_error(y_train, pred)
    })

history_df = pd.DataFrame(history)
history_df.tail(5).round(2)

### 손실 곡선과 가중치 세 개

**나와야 하는 것:** 4000 근처에서 뚝 떨어져 바닥에 붙는 손실 곡선, 그리고 선 세 개. `study`가 10 근처로 가장 높이 오르고, `count`는 6 가까이 올랐다가 3.3으로 내려와요.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    history_df["epoch"],
    history_df["mse"]
)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_df["epoch"], history_df["w1"], label="study")
ax.plot(history_df["epoch"], history_df["w2"], label="sleep")
ax.plot(history_df["epoch"], history_df["w3"], label="count")

ax.set_xlabel("Epoch")
ax.set_ylabel("Weight")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 답과 비교

**나와야 하는 것:** `[9.97 3.63 3.24] 67.43`. 500 epoch의 SGD(9.93, 3.62, 3.29, 67.42)가 이 답 가까이 와 있어요. 절편 67.43은 학습용 7명의 평균 점수예요.

In [ ]:
exact = LinearRegression()
exact.fit(X_train_scaled, y_train)

print(exact.coef_.round(2), round(exact.intercept_, 2))
print("학습용 평균 점수", round(y_train.mean(), 2))

## 7. KNN 준비: 얼굴 사진 한 장은 0~1 숫자 4096개

### 얼굴 사진 400장 받기

**나와야 하는 것:** 처음에는 `downloading Olivetti faces ...`가 한 줄 나오고, 그다음부터는 아무것도 안 나와요(저장해 둔 것을 읽어요).

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

# ./datasets 폴더에 받아 두고 다음부터는 거기서 읽는다
faces = fetch_olivetti_faces(data_home="./datasets")

### 크기 보기

**나와야 하는 것:** `(400, 64, 64)`, `(400,)`, 사람 번호 `0`~`39`, 그리고 `(400, 4096)`

In [ ]:
print("사진 묶음 크기 : ", faces.images.shape)
print("정답 묶음 크기 : ", faces.target.shape)
print("사람 종류 : ", np.unique(faces.target))

# 사진 한 장을 한 줄로 편 것
print("한 줄로 편 크기 : ", faces.data.shape)

### 0번 사진

**나와야 하는 것:** 흑백 얼굴 사진 한 장과 제목 `Person 0`

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(faces.images[0], cmap="gray", vmin=0, vmax=1)
plt.title(f"Person {faces.target[0]}")
plt.axis("off")
plt.show()

### 사진은 숫자 표

**나와야 하는 것:** 8행 8열 숫자 표의 첫 줄 `[0.57 0.58 0.61 0.57 0.54 0.55 0.59 0.64]`, `가장 어두운 값:  0.0`, `가장 밝은 값:  1.0`, 그리고 칸마다 숫자가 적힌 회색 격자 그림

In [ ]:
# 0번 사진의 20~27행, 20~27열
corner = faces.images[0][20:28, 20:28]
print(np.round(corner, 2))

print("가장 어두운 값: ", faces.images.min())
print("가장 밝은 값: ", faces.images.max())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(corner, cmap="gray", vmin=0, vmax=1)

for row in range(8):
    for col in range(8):
        value = corner[row, col]
        # 가로 위치가 열(col), 세로 위치가 행(row)
        ax.text(col, row, f"{value:.1f}", ha="center", va="center")

plt.show()

### 한 사람의 사진 10장

수업 코드는 그리는 줄까지 치고 끝났어요. 칸마다 축을 끄고 `plt.show()`를 붙여 마무리했어요.

**나와야 하는 것:** 같은 사람의 얼굴 10장이 한 줄로. 표정과 고개 방향이 조금씩 달라요.

In [ ]:
def show_person(person):
    # 그 사람의 사진만 고른다
    pics = faces.images[faces.target == person]

    fig, axes = plt.subplots(1, 10, figsize=(15, 2))
    for ax, img in zip(axes, pics):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    fig.suptitle(f"Person {person}")
    plt.show()

show_person(0)

## 혼자 치기

위 칸을 모두 실행한 뒤에 풀어요(`df`, `X`, `y`, `features`, `model`, `result_df`, `scaler`, `faces`를 그대로 써요). 정답은 맨 아래에 있어요.

**문제 1.** 2절의 `model`로 공부 5시간, 수면 8시간, 문제 30개인 학생의 점수를 예측해 소수 둘째 자리로 찍어 보세요.

In [ ]:
# 여기에 써 보세요

**문제 2.** 3절의 `result_df`에서 예측오차의 절댓값이 가장 큰 학생의 행 번호를 찾아보세요.

In [ ]:
# 여기에 써 보세요

**문제 3.** `random_state=0`으로 7명, 3명을 다시 나눈 뒤 `LinearRegression`으로 학습하고, 시험용의 MAE를 소수 둘째 자리로 찍어 보세요. 4절의 3.09와 비교해 보세요.

In [ ]:
# 여기에 써 보세요

**문제 4.** `StandardScaler`로 공부 시간 `[1, 2, 3, 4, 5]` 다섯 개만 표준화해 소수 둘째 자리로 찍어 보세요. (힌트: 입력은 `[[1], [2], [3], [4], [5]]`처럼 2차원)

In [ ]:
# 여기에 써 보세요

**문제 5.** `faces`에서 사람 3의 사진이 몇 장인지, 그 사진들의 픽셀 평균 밝기가 얼마인지 소수 둘째 자리로 찍어 보세요.

In [ ]:
# 여기에 써 보세요

## 정답

먼저 풀어 본 뒤에 봐요.

문제 1

```python
new_student = pd.DataFrame({
    "study_hours": [5],
    "sleep_hours": [8],
    "practice_count": [30]
})
print(round(model.predict(new_student)[0], 2))
# 87.09
```

문제 2

```python
print(result_df["예측오차"].abs().idxmax())
# 1
```

문제 3

```python
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0
)
other = LinearRegression()
other.fit(X_tr, y_tr)
print(round(mean_absolute_error(y_te, other.predict(X_te)), 2))
# 1.46
```

문제 4

```python
hours = StandardScaler().fit_transform([[1], [2], [3], [4], [5]])
print(hours.ravel().round(2))
# [-1.41 -0.71  0.    0.71  1.41]
```

문제 5

```python
pics = faces.images[faces.target == 3]
print(len(pics))
print(round(float(pics.mean()), 2))
# 10
# 0.57
```